# Relive — Phase 0 de-risk notebook (InstantSplat on Colab)

**Goal:** phone video *or* a folder of photos → Gaussian Splatting `.ply` you can walk around in a headset.

**Path:** frames → MASt3R poses + sparse cloud → 3DGS training → `.ply` (InstantSplat).

**Runtime:** set Colab to a **GPU** runtime (Runtime → Change runtime type → T4 GPU). Free T4 = 16GB VRAM — keep the capture small (this notebook defaults to ~12 frames at 960px).

**Confidence notes:** cells 1–5 and the viewing instructions are solid. Cell 6 (the InstantSplat run) mirrors the repo's own `run_infer.sh`; research repos usually need ONE path/flag tweak on first run. If it errors, read the error — it's almost always “images not found at path X”, which is a one-line fix.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('Torch:', torch.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch to a GPU runtime')

## 2. Clone InstantSplat + download the MASt3R checkpoint

In [ ]:
%cd /content
![ -d InstantSplat ] || git clone --recursive https://github.com/NVlabs/InstantSplat.git
%cd /content/InstantSplat
!mkdir -p mast3r/checkpoints/
![ -f mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth ] || \
  wget -q --show-progress https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth -P mast3r/checkpoints/
print('done')

## 3. Install dependencies (compiles the CUDA splat kernels — takes a few minutes)
Colab already ships a CUDA-matched PyTorch, so we skip conda and pip-install on top of it. If a submodule fails to compile, re-run this cell once (transient) before debugging.

In [ ]:
%cd /content/InstantSplat
!pip install -q -r requirements.txt
# custom CUDA kernels (the actual gaussian splat rasterizer):
!pip install -q submodules/simple-knn
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/fused-ssim
# NOTE: we deliberately SKIP the optional 'curope' CUDA kernel.
# Colab's newer PyTorch rejects its old dispatch API (kernels.cu compile error),
# and CroCo/MASt3R auto-falls back to a pure-PyTorch RoPE when curope is absent
# (prints a "using a slow pytorch version" warning) — fine for a few sparse images.
print('install complete (curope intentionally skipped — pytorch RoPE fallback will be used)')

## 4. Provide your input
Run the cell, then **upload either**:
- one video file (`.mp4`/`.mov`), **or**
- several photos (`.jpg`/`.png`) — e.g. old pictures of a place.

For a video: record a slow, steady sweep of one room, ~20–30s, good light. For photos: the more overlap between them, the better.

In [ ]:
import os, glob
from google.colab import files

INPUT_DIR = '/content/input'
os.makedirs(INPUT_DIR, exist_ok=True)
for f in glob.glob(INPUT_DIR + '/*'):
    os.remove(f)
%cd /content/input
print('Upload a video OR a set of images:')
uploaded = files.upload()
%cd /content/InstantSplat
print('Received:', list(uploaded.keys()))

## 5. Extract sharp, well-spread frames (drops motion-blurred ones)
Uses variance-of-Laplacian to score sharpness. From a video it samples candidates across the whole clip, keeps the sharpest `TARGET_FRAMES`, and downsizes to `MAX_WIDTH`. From photos it just sharpness-culls and resizes. Tune `TARGET_FRAMES` up later once you know the T4 holds; start small.

In [ ]:
import cv2, os, glob, numpy as np, shutil

# ---- tunables ----
TARGET_FRAMES = 12      # keep small on a T4; sparse-view method is happy with few
MAX_WIDTH     = 960     # downsize long edge to control VRAM
CANDIDATE_MULT = 4      # for video: score this many x TARGET candidates, keep sharpest
# ------------------

SCENE = 'relive'
SCENE_DIR = f'/content/InstantSplat/assets/{SCENE}/images'
os.makedirs(SCENE_DIR, exist_ok=True)
for f in glob.glob(SCENE_DIR + '/*'):
    os.remove(f)

def sharpness(img):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(g, cv2.CV_64F).var()

def resize(img):
    h, w = img.shape[:2]
    if w > MAX_WIDTH:
        img = cv2.resize(img, (MAX_WIDTH, int(h * MAX_WIDTH / w)), interpolation=cv2.INTER_AREA)
    return img

inputs = sorted(glob.glob('/content/input/*'))
vids = [f for f in inputs if f.lower().endswith(('.mp4', '.mov', '.avi', '.mkv', '.webm'))]
imgs = [f for f in inputs if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

candidates = []  # list of (sharpness, image_array)

if vids:
    cap = cv2.VideoCapture(vids[0])
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    n_cand = min(max(TARGET_FRAMES * CANDIDATE_MULT, TARGET_FRAMES), max(total, 1))
    idxs = np.linspace(0, max(total - 1, 0), n_cand).astype(int)
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        if ok:
            candidates.append((sharpness(frame), i, resize(frame)))
    cap.release()
    # keep sharpest TARGET, then re-sort by time so coverage stays even
    candidates.sort(key=lambda t: t[0], reverse=True)
    kept = sorted(candidates[:TARGET_FRAMES], key=lambda t: t[1])
    kept = [(s, im) for s, _, im in kept]
elif imgs:
    scored = [(sharpness(cv2.imread(f)), cv2.imread(f)) for f in imgs]
    scored.sort(key=lambda t: t[0], reverse=True)
    kept = [(s, resize(im)) for s, im in scored[:max(TARGET_FRAMES, len(scored))]]
else:
    raise RuntimeError('No video or images found in /content/input — re-run cell 4.')

for n, (s, im) in enumerate(kept):
    cv2.imwrite(f'{SCENE_DIR}/{n:04d}.jpg', im, [cv2.IMWRITE_JPEG_QUALITY, 95])

N_VIEWS = len(kept)
print(f'Wrote {N_VIEWS} frames to {SCENE_DIR}')
print('Sharpness range:', round(min(s for s,_ in kept),1), '→', round(max(s for s,_ in kept),1))

## 6. Run InstantSplat (poses → 3DGS → `.ply`)
This runs the three stages the repo's `run_infer.sh` runs: `init_geo.py` (MASt3R poses + sparse cloud) → `train.py` (Gaussian training) → `render.py`. We point it at our `relive` scene and our frame count.

⚠️ **If this errors, read it and send it to me.** The likely one is a path mismatch (the script expecting images at a slightly different folder). We reproduce the repo's own commands here to minimise that.

In [ ]:
import os, glob, shutil, re
%cd /content/InstantSplat

# FIX 1: PyTorch >=2.6 torch.load(weights_only=True) rejects the MASt3R checkpoint. Restore old behaviour.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
for path in ['mast3r/model.py', 'dust3r/model.py']:
    p = f'/content/InstantSplat/{path}'
    if os.path.exists(p):
        s = open(p).read()
        s2 = re.sub(r"torch\.load\(([^)\n]*map_location[^)\n]*)\)",
                    lambda m: m.group(0) if 'weights_only' in m.group(1)
                              else f"torch.load({m.group(1)}, weights_only=False)", s)
        if s2 != s:
            open(p, 'w').write(s2); print('patched torch.load in', path)

SCENE = 'relive'
SOURCE_PATH = f'/content/InstantSplat/assets/{SCENE}/'
IMAGE_DIR   = SOURCE_PATH + 'images'
for d in glob.glob(SOURCE_PATH + 'sparse_*') + glob.glob(IMAGE_DIR + '/sparse_*'):
    shutil.rmtree(d, ignore_errors=True)

N_VIEWS = len([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
GS_TRAIN_ITER = 1000
MODEL_PATH = f'/content/InstantSplat/output_infer/{SCENE}/{N_VIEWS}_views'
os.makedirs(MODEL_PATH, exist_ok=True)
print('Source:', SOURCE_PATH, '| views:', N_VIEWS, '| iters:', GS_TRAIN_ITER)

# Stage 1 — geometry init. FIX 2: dropped --infer_video (its pose interpolation runs BEFORE the
# sparse-file save and was getting killed, so points3D.bin/images.txt never got written).
!python -W ignore ./init_geo.py -s "$SOURCE_PATH" -m "$MODEL_PATH" --n_views $N_VIEWS --focal_avg --co_vis_dsp --conf_aware_ranking

# Stage 2 — train the gaussians (this is what produces point_cloud.ply).
!python ./train.py -s "$SOURCE_PATH" -m "$MODEL_PATH" -r 1 --n_views $N_VIEWS --iterations $GS_TRAIN_ITER --pp_optimizer --optim_pose

# Stage 3 — render training views (optional sanity check; .ply already exists after Stage 2). --infer_video dropped.
!python ./render.py -s "$SOURCE_PATH" -m "$MODEL_PATH" -r 1 --n_views $N_VIEWS --iterations $GS_TRAIN_ITER

print('\n--- run finished; look for point_cloud.ply below ---')

## 7. Find and download the `.ply`

In [ ]:
import glob
from google.colab import files

plys = sorted(glob.glob('/content/InstantSplat/output_infer/**/point_cloud*.ply', recursive=True))
print('Found .ply files:')
for p in plys:
    print('  ', p)
if plys:
    latest = plys[-1]
    print('\nDownloading:', latest)
    files.download(latest)
else:
    print('\nNo .ply found — check the cell 6 output/logs for where training wrote its result.')

## 8. View it (no install, dodges your weak PC GPU)
1. **Desktop sanity check first:** open the SuperSplat viewer — https://superspl.at/editor — and drag your `.ply` in. Does the room look like a real place, or blobby/holey? This is checkpoint #1 (5 minutes).
2. **The actual gate — in the headset:** load the same splat in a WebXR viewer and open it in the **Quest browser** (renders on the Quest's own GPU, so your PC's GTX is irrelevant). Put the headset on someone who wasn't involved and watch their reaction.

**Scale:** InstantSplat/MASt3R output is roughly metric but not exact — if the room feels giant/tiny in the headset, that's the known scale risk, not a bug. We solve it properly later (ARKit metric poses). For now, note it and move on.

**Gotchas:** textureless walls / windows / reflections will be the ugliest areas, and quality degrades as you move away from where the photos were taken — keep the demo viewpoint near the captured path.